# NB23 · Decorators

`advanced/` · notebook 23 of 32 · 36 questions + a mini-project

A decorator is a function that takes a function and gives back a
replacement for it. You already have every piece: functions are values
(NB09), closures remember variables from the enclosing call (NB09), and
`lru_cache` showed you a decorator in action (NB11). This notebook shows you
how the `@` line works and how to write your own.

The whole idea fits in one line: writing `@deco` above `def f` means
`f = deco(f)`. Everything else here (wrappers, `functools.wraps`,
decorators that take arguments, decorators written as classes) builds on
that one rewrite.

**About timing.** Real timing decorators call `time.perf_counter()`. Real
times change on every run, so these questions hand the decorator a *fake
clock* instead: a function that returns made-up times in a fixed order.
This is also how seniors test timing code.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB09: closures
- [ ] NB22: generators

## What you'll practise

- How @ works; writing a decorator
- functools.wraps
- Decorators that take arguments (factories); stacking order
- Decorating methods; class decorators
- Registration decorators (plugin registries)
- lru_cache and cache_info; singledispatch
- Decorators written as classes; validating arguments with inspect.signature

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L1 · Extremely Easy

### NB23-Q01 · L1 · How @ works · Write

Write a decorator `announce` that prints `Running: <function name>` and
then calls the function. Use it with `@` on a function `backup()` that
prints `backing up files`. Call `backup()` once.

```text
Running: backup
backing up files
```

<details><summary><b>Hint</b></summary>

Inside `announce(func)`, define an inner function `wrapper()` that prints and then calls `func()`. Return `wrapper` (the function itself, no brackets). Every function knows its own name in `func.__name__`.

</details>

<details><summary><b>Expected output</b></summary>

```text
Running: backup
backing up files
```

</details>

<details><summary><b>Solution</b></summary>

```python
def announce(func):
    def wrapper():
        print("Running:", func.__name__)
        func()
    return wrapper

@announce
def backup():
    print("backing up files")

backup()
```

`@announce` above `def backup` means `backup = announce(backup)`. From then on, the name `backup` points at `wrapper`. Calling it prints the message, then calls the original function, which `wrapper` still remembers through its closure (NB09).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import wraps

def announce(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Running: {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@announce
def backup():
    print("backing up files")

backup()
```

Three habits seniors put in every decorator: `*args, **kwargs` so it works on any function, `return func(...)` so the result isn't lost, and `@wraps(func)` so the wrapper keeps the original name and docstring (Q04 shows what goes wrong without it).

</details>

In [ ]:
# your code here


### NB23-Q02 · L1 · How @ works · Predict

When does a decorator run? Predict the output.

**What does this print?**

```python
def register(func):
    print("registering", func.__name__)
    return func

@register
def hello():
    print("hello!")

print("start")
hello()
hello()
```

<details><summary><b>Hint</b></summary>

`@register` is the same as writing `hello = register(hello)` right after the `def`. When does that line run?

</details>

<details><summary><b>Expected output</b></summary>

```text
registering hello
start
hello!
hello!
```

</details>

<details><summary><b>Solution</b></summary>

The decorator runs **once, when the `def` runs**, not when the function is called. So `registering hello` appears first, before `start`. `register` hands back the original function unchanged, so the two calls print `hello!` with nothing extra.

</details>

<details><summary><b>Senior dev solution</b></summary>

Running at definition time is what makes registration decorators work (Q18 and the mini-project): plugins and URL routes get recorded the moment the module is imported, with no extra setup call.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB23-Q03 · L1 · How @ works · Write

`double_result` is a decorator and `price` is a plain function. **Without**
using the `@` syntax, make the name `price` refer to the decorated
version, then print `price(4)`.

**Starting code (already in the cell below):**

```python
def double_result(func):
    def wrapper(qty):
        return func(qty) * 2
    return wrapper

def price(qty):
    return qty * 3
```

<details><summary><b>Hint</b></summary>

`@double_result` above `def price` is shorthand for one assignment. Write that assignment yourself.

</details>

<details><summary><b>Expected output</b></summary>

```text
24
```

</details>

<details><summary><b>Solution</b></summary>

```python
price = double_result(price)
print(price(4))
```

`double_result(price)` builds a new function (`wrapper`) that calls the original `price` and doubles the answer. Assigning it back to the name `price` is exactly what `@` does. `4 * 3 = 12`, doubled is `24`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def double_result(func):
    def wrapper(qty):
        return func(qty) * 2
    return wrapper

@double_result
def price(qty):
    return qty * 3

print(price(4))
```

`@` is pure shorthand, but it puts the decoration next to the `def`, where a reader sees it first. The manual form is still useful when you want both versions: `fast_price = double_result(price)` keeps the original too.

</details>

In [ ]:
def double_result(func):
    def wrapper(qty):
        return func(qty) * 2
    return wrapper

def price(qty):
    return qty * 3

# your code here


### NB23-Q04 · L1 · functools.wraps · Predict

A decorator without `functools.wraps`. Predict the output.

**What does this print?**

```python
def log_call(func):
    def wrapper(*args, **kwargs):
        print("calling", func.__name__)
        return func(*args, **kwargs)
    return wrapper

@log_call
def add(a, b):
    """Add two numbers."""
    return a + b

print(add.__name__)
print(add.__doc__)
```

<details><summary><b>Hint</b></summary>

After decoration, which function does the name `add` actually point to? What is *that* function's name, and does it have a docstring?

</details>

<details><summary><b>Expected output</b></summary>

```text
wrapper
None
```

</details>

<details><summary><b>Solution</b></summary>

The name `add` now points at `wrapper`, so `add.__name__` is `wrapper` and `add.__doc__` is `None` (wrapper has no docstring). The original name and docstring are hidden. That breaks `help(add)`, error messages, logs and any tool that looks functions up by name.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import wraps

def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("calling", func.__name__)
        return func(*args, **kwargs)
    return wrapper

@log_call
def add(a, b):
    """Add two numbers."""
    return a + b

print(add.__name__)
print(add.__doc__)
```

`@wraps(func)` copies `__name__`, `__doc__`, `__module__` and a few more onto the wrapper, and sets `add.__wrapped__` to the original function. This version prints `add` and `Add two numbers.`. Seniors never leave it out.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L2 · Easy

### NB23-Q05 · L2 · Writing a decorator · Write

Write a decorator `shout` that turns a function's returned string into
upper case. Use it on `greet(name)`, which returns `"hello, " + name`,
and print `greet("ada")`.

```text
HELLO, ADA
```

<details><summary><b>Hint</b></summary>

The wrapper needs to accept the same argument as `greet`, call the original, and **return** the changed result.

</details>

<details><summary><b>Expected output</b></summary>

```text
HELLO, ADA
```

</details>

<details><summary><b>Solution</b></summary>

```python
def shout(func):
    def wrapper(name):
        return func(name).upper()
    return wrapper

@shout
def greet(name):
    return "hello, " + name

print(greet("ada"))
```

The wrapper calls the original `greet`, gets `"hello, ada"` back, and returns `.upper()` of it. A decorator can change what goes *in* (the arguments), what comes *out* (the result), or both.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import wraps

def shout(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper

@shout
def greet(name):
    return f"hello, {name}"

print(greet("ada"))
```

`wrapper(name)` only fits functions with exactly one argument. With `*args, **kwargs` the same `shout` works on `greet(name, punctuation="!")` or anything else that returns a string.

</details>

In [ ]:
# your code here


### NB23-Q06 · L2 · functools.wraps · Fix

`audit` should print the call **and** let the result through, so this
should print `audit: total` and then `60`. It prints `None` instead of
`60`. Fix it.

**Buggy code (copied into the cell below):**

```python
from functools import wraps

def audit(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("audit:", func.__name__)
        func(*args, **kwargs)
    return wrapper

@audit
def total(prices):
    return sum(prices)

print(total([10, 20, 30]))
```

<details><summary><b>Hint</b></summary>

The original function returns a value. Where does that value go inside the wrapper?

</details>

<details><summary><b>Expected output</b></summary>

```text
audit: total
60
```

</details>

<details><summary><b>Solution</b></summary>

```python
from functools import wraps

def audit(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("audit:", func.__name__)
        return func(*args, **kwargs)
    return wrapper

@audit
def total(prices):
    return sum(prices)

print(total([10, 20, 30]))
```

The wrapper called `func` but threw its result away. A function without `return` gives back `None`, so every decorated call returned `None`. The wrapper *replaces* the function, so it must `return` whatever the original returns.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import wraps

def audit(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print("audit:", func.__name__)
        return result
    return wrapper

@audit
def total(prices):
    return sum(prices)

print(total([10, 20, 30]))
```

Seniors store the result in a name when the wrapper has work to do after the call (logging, timing, checking). The audit line could now include the result too. Forgetting `return` is the most common decorator bug, and it fails silently.

</details>

In [ ]:
from functools import wraps

def audit(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("audit:", func.__name__)
        func(*args, **kwargs)
    return wrapper

@audit
def total(prices):
    return sum(prices)

print(total([10, 20, 30]))


### NB23-Q07 · L2 · Writing a decorator · Predict

One counting decorator, used on two functions. Predict the output.

**What does this print?**

```python
def count_calls(func):
    count = 0
    def wrapper(*args, **kwargs):
        nonlocal count
        count += 1
        print(f"{func.__name__} call #{count}")
        return func(*args, **kwargs)
    return wrapper

@count_calls
def save():
    return None

@count_calls
def load():
    return None

save()
save()
load()
save()
```

<details><summary><b>Hint</b></summary>

Each `@count_calls` line calls `count_calls` once. Does each call get its own `count` variable?

</details>

<details><summary><b>Expected output</b></summary>

```text
save call #1
save call #2
load call #1
save call #3
```

</details>

<details><summary><b>Solution</b></summary>

Every call to `count_calls` creates a fresh local `count` and a fresh `wrapper` that closes over it. `save` and `load` each have their own counter, so `load` starts at `#1` while `save` reaches `#3`.

</details>

<details><summary><b>Senior dev solution</b></summary>

This is closure state (NB09) doing the job a class attribute might otherwise do. If the count had been a global variable, both functions would share it, which is almost never what you want.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB23-Q08 · L2 · Writing a decorator · Write

Write a decorator `trace` that prints each call with its arguments and
its result, then returns the result. Positional arguments appear as
their `repr()`, keyword arguments as `name=repr(value)`, all separated
by `, `. Use it on `area(width, height)` and make these two calls:
`area(3, 4)` and `area(3, height=4)`.

```text
area(3, 4) -> 12
area(3, height=4) -> 12
```

<details><summary><b>Hint</b></summary>

`args` is a tuple and `kwargs` is a dict. Build one list of strings from both, then `", ".join(...)` it.

</details>

<details><summary><b>Expected output</b></summary>

```text
area(3, 4) -> 12
area(3, height=4) -> 12
```

</details>

<details><summary><b>Solution</b></summary>

```python
def trace(func):
    def wrapper(*args, **kwargs):
        parts = [repr(a) for a in args]
        parts += [f"{key}={value!r}" for key, value in kwargs.items()]
        result = func(*args, **kwargs)
        print(f"{func.__name__}({', '.join(parts)}) -> {result!r}")
        return result
    return wrapper

@trace
def area(width, height):
    return width * height

area(3, 4)
area(3, height=4)
```

`*args` collects the positional values as a tuple and `**kwargs` collects the keyword ones as a dict. Passing them on with `func(*args, **kwargs)` unpacks them again, so the original sees exactly the call that was made. `!r` in an f-string uses `repr()`, which puts quotes around strings.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import wraps

def trace(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        parts = [*map(repr, args), *(f"{k}={v!r}" for k, v in kwargs.items())]
        result = func(*args, **kwargs)
        print(f"{func.__name__}({', '.join(parts)}) -> {result!r}")
        return result
    return wrapper

@trace
def area(width, height):
    return width * height

area(3, 4)
area(3, height=4)
```

Same idea with `wraps` and one unpacked list. Printing *after* the call means a crashing call prints nothing; in real tracing seniors log the call first and the result second, or use `logging` (NB28) at `DEBUG` level so it can be switched off.

</details>

In [ ]:
# your code here


### NB23-Q09 · L2 · How @ works · Error

This decorator is missing one line. Which error does calling `greet()` raise?

**Which error does this raise?**

```python
def announce(func):
    def wrapper():
        print("about to greet")
        return func()

@announce
def greet():
    return "hi"

print(greet())
```

<details><summary><b>Hint</b></summary>

What does `announce` return? Then what does the name `greet` point to?

</details>

<details><summary><b>Expected output</b></summary>

```text
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError`: `announce` never returns `wrapper`, so it returns `None`, and `greet = announce(greet)` makes `greet` equal to `None`. Calling `None()` fails with "'NoneType' object is not callable". The decoration itself worked; the crash comes at the first call.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def announce(func):
    def wrapper():
        print("about to greet")
        return func()
    return wrapper

@announce
def greet():
    return "hi"

print(greet())
```

When a decorated function is suddenly `None` or the wrong thing, check the two `return` lines first: the decorator must return the wrapper, and the wrapper must return the result.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB23-Q10 · L2 · Stacking order · Predict

Two decorators stacked on one function. Predict the output.

**What does this print?**

```python
def bold(func):
    def wrapper():
        return "<b>" + func() + "</b>"
    return wrapper

def italic(func):
    def wrapper():
        return "<i>" + func() + "</i>"
    return wrapper

@bold
@italic
def title():
    return "Sale"

print(title())
```

<details><summary><b>Hint</b></summary>

Stacked decorators apply from the bottom up: `title = bold(italic(title))`.

</details>

<details><summary><b>Expected output</b></summary>

```text
<b><i>Sale</i></b>
```

</details>

<details><summary><b>Solution</b></summary>

`italic` is closest to the function, so it wraps it first. `bold` then wraps *that*. When you call `title()`, the outer `bold` wrapper calls the `italic` wrapper, which calls the original. The result is `<b><i>Sale</i></b>`.

</details>

<details><summary><b>Senior dev solution</b></summary>

Read a stack of decorators like layers of an onion: the top one is the outermost layer, the one a caller hits first. Swapping the two lines gives `<i><b>Sale</b></i>`.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB23-Q11 · L2 · Decorator factories · Write

Write `prefix(text)`, a decorator **that takes an argument**. It adds
`text` to the front of the decorated function's returned string. Use
it as `@prefix("[WARN] ")` on `disk_message()`, which returns
`"disk almost full"`, and print the call.

```text
[WARN] disk almost full
```

<details><summary><b>Hint</b></summary>

`@prefix("[WARN] ")` first *calls* `prefix("[WARN] ")`. That call must return a normal decorator, so you need three levels: `prefix` → `decorator(func)` → `wrapper()`.

</details>

<details><summary><b>Expected output</b></summary>

```text
[WARN] disk almost full
```

</details>

<details><summary><b>Solution</b></summary>

```python
def prefix(text):
    def decorator(func):
        def wrapper():
            return text + func()
        return wrapper
    return decorator

@prefix("[WARN] ")
def disk_message():
    return "disk almost full"

print(disk_message())
```

The line `@prefix("[WARN] ")` runs `prefix("[WARN] ")` first, which returns `decorator`. Then `decorator` is applied to `disk_message` as usual. `wrapper` can see both `text` and `func` through two layers of closure. A function that builds decorators is called a *decorator factory*.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import wraps

def prefix(text):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            return f"{text}{func(*args, **kwargs)}"
        return wrapper
    return decorator

@prefix("[WARN] ")
def disk_message():
    return "disk almost full"

print(disk_message())
```

Seniors name the three levels `factory → decorator → wrapper` consistently, so anyone reading the code knows which layer they're in. Here `wraps` goes on the innermost function, the one that replaces the original.

</details>

In [ ]:
# your code here


---

## L3 · Intermediate

### NB23-Q12 · L3 · Stacking order · Predict

Both decorators print once when they are **applied** and twice when the
function is **called**. Predict the full output.

**What does this print?**

```python
from functools import wraps

def first(func):
    print("applying first")
    @wraps(func)
    def wrapper():
        print("first: before")
        func()
        print("first: after")
    return wrapper

def second(func):
    print("applying second")
    @wraps(func)
    def wrapper():
        print("second: before")
        func()
        print("second: after")
    return wrapper

@first
@second
def job():
    print("job runs")

print("--- calling ---")
job()
```

<details><summary><b>Hint</b></summary>

Applying happens bottom-up, when the `def` runs. Calling goes top-down into the layers and then back out.

</details>

<details><summary><b>Expected output</b></summary>

```text
applying second
applying first
--- calling ---
first: before
second: before
job runs
second: after
first: after
```

</details>

<details><summary><b>Solution</b></summary>

`job = first(second(job))`, so `second` is applied first and prints `applying second`, then `first`. When called, the outer `first` wrapper runs first, steps into `second`'s wrapper, which steps into `job`. On the way back out, `second: after` comes before `first: after`. Applied bottom-up, run top-down.

</details>

<details><summary><b>Senior dev solution</b></summary>

This is why order matters in real stacks. `@login_required` above `@cache` checks the login before the cache is consulted; the other way round, a cached page could be served to anyone (Q28 is this bug).

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB23-Q13 · L3 · Decorator factories · Write

`fetch_rates` fails twice with `ConnectionError` and then works. Write a
decorator factory `retry(times)` whose wrapper calls the function up to
`times` times. After each `ConnectionError` it prints the attempt
number and the error message. If the last attempt also fails, it
re-raises the error.

Apply it with `fetch_rates = retry(times=3)(fetch_rates)` (exactly what
`@retry(times=3)` would do), then print `fetch_rates()`.

```text
attempt 1 failed: network down
attempt 2 failed: network down
rates ok
```

**Starting code (already in the cell below):**

```python
calls = {"count": 0}

def fetch_rates():
    calls["count"] += 1
    if calls["count"] < 3:
        raise ConnectionError("network down")
    return "rates ok"
```

<details><summary><b>Hint</b></summary>

Loop over `range(1, times + 1)`. Inside, `try` to `return func(...)`. In the `except ConnectionError as e` branch, print, and if this was the last attempt, `raise`.

</details>

<details><summary><b>Expected output</b></summary>

```text
attempt 1 failed: network down
attempt 2 failed: network down
rates ok
```

</details>

<details><summary><b>Solution</b></summary>

```python
def retry(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except ConnectionError as e:
                    print(f"attempt {attempt} failed: {e}")
                    if attempt == times:
                        raise
        return wrapper
    return decorator

fetch_rates = retry(times=3)(fetch_rates)
print(fetch_rates())
```

`return` inside the `try` leaves the loop as soon as a call succeeds. A failure prints and goes round again. On the final attempt, a bare `raise` re-raises the same `ConnectionError` (NB12), so the caller still finds out that everything failed instead of getting a silent `None`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import wraps

def retry(times, exceptions=(ConnectionError,)):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"attempt {attempt} failed: {e}")
                    if attempt == times:
                        raise
        return wrapper
    return decorator

fetch_rates = retry(times=3)(fetch_rates)
print(fetch_rates())
```

A senior makes the retryable exceptions a parameter. Retrying a `ValueError` is pointless (bad data stays bad), while network errors are often temporary. Real retry code also waits between attempts; Q34 adds that, with a fake sleep.

</details>

In [ ]:
calls = {"count": 0}

def fetch_rates():
    calls["count"] += 1
    if calls["count"] < 3:
        raise ConnectionError("network down")
    return "rates ok"

# your code here


### NB23-Q14 · L3 · @deco vs @deco() · Error

`log_call` is a plain decorator, but someone used it with brackets.
Which error does this raise?

**Which error does this raise?**

```python
def log_call(func):
    def wrapper(*args, **kwargs):
        print("calling", func.__name__)
        return func(*args, **kwargs)
    return wrapper

@log_call()
def report():
    return "done"

print(report())
```

<details><summary><b>Hint</b></summary>

`@log_call()` calls `log_call()` first, before it ever sees `report`. What does `log_call` need?

</details>

<details><summary><b>Expected output</b></summary>

```text
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError`: `@log_call()` means `report = log_call()(report)`. The first call, `log_call()`, has no argument for `func`, so Python raises a `TypeError` about a missing positional argument. It fails while the `def` runs, before `report()` is ever called.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def log_call(func):
    def wrapper(*args, **kwargs):
        print("calling", func.__name__)
        return func(*args, **kwargs)
    return wrapper

@log_call
def report():
    return "done"

print(report())
```

Brackets or no brackets depends on how the decorator was written: plain decorators take `func`, factories take settings and return a decorator. Q27 shows how to write one that accepts both forms.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB23-Q15 · L3 · @deco vs @deco() · Fix

`repeat(times)` is a decorator factory. This should print
`Hello, Ada!` three times, but it prints nothing at all. Fix it.

**Buggy code (copied into the cell below):**

```python
def repeat(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for _ in range(times):
                func(*args, **kwargs)
        return wrapper
    return decorator

@repeat
def greet(name):
    print(f"Hello, {name}!")

greet("Ada")
```

<details><summary><b>Hint</b></summary>

Work out what `@repeat` without brackets does: `greet = repeat(greet)`. What is `greet` now, and what does calling it with `"Ada"` do?

</details>

<details><summary><b>Expected output</b></summary>

```text
Hello, Ada!
Hello, Ada!
Hello, Ada!
```

</details>

<details><summary><b>Solution</b></summary>

```python
def repeat(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for _ in range(times):
                func(*args, **kwargs)
        return wrapper
    return decorator

@repeat(3)
def greet(name):
    print(f"Hello, {name}!")

greet("Ada")
```

Without brackets, `greet = repeat(greet)`, so `times` is the function and `greet` becomes `decorator`. Then `greet("Ada")` runs `decorator("Ada")`, which builds a wrapper and returns it without calling anything. No error, no output: the worst kind of bug. `@repeat(3)` gives the factory its argument first.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import wraps

def repeat(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                func(*args, **kwargs)
        return wrapper
    return decorator

@repeat(times=3)
def greet(name):
    print(f"Hello, {name}!")

greet("Ada")
```

`@repeat(times=3)` reads better than `@repeat(3)` and makes the mistake harder to make. A defensive factory can also check its input: `if callable(times): raise TypeError("use @repeat(n)")` turns the silent bug into a clear error.

</details>

In [ ]:
def repeat(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for _ in range(times):
                func(*args, **kwargs)
        return wrapper
    return decorator

@repeat
def greet(name):
    print(f"Hello, {name}!")

greet("Ada")


### NB23-Q16 · L3 · Decorating methods · Write

Write a decorator `log_method` for **methods**. Before calling the method
it prints `<ClassName>.<method name>(<args>)`, with the arguments (not
`self`) as their `repr()` separated by `, `. Use it on `deposit` and
`withdraw` in this `Account` class, then run the calls shown and print
the balance.

```py
class Account:
    def __init__(self, balance):
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        self.balance -= amount
```

Calls: `acct = Account(100)`, `acct.deposit(50)`, `acct.withdraw(30)`,
then print `acct.balance`.

```text
Account.deposit(50)
Account.withdraw(30)
120
```

<details><summary><b>Hint</b></summary>

When a method is called, `self` arrives as the **first** argument to the wrapper. Give the wrapper `(self, *args, **kwargs)` and pass all of them on.

</details>

<details><summary><b>Expected output</b></summary>

```text
Account.deposit(50)
Account.withdraw(30)
120
```

</details>

<details><summary><b>Solution</b></summary>

```python
def log_method(func):
    def wrapper(self, *args, **kwargs):
        shown = ", ".join(repr(a) for a in args)
        print(f"{type(self).__name__}.{func.__name__}({shown})")
        return func(self, *args, **kwargs)
    return wrapper

class Account:
    def __init__(self, balance):
        self.balance = balance

    @log_method
    def deposit(self, amount):
        self.balance += amount

    @log_method
    def withdraw(self, amount):
        self.balance -= amount

acct = Account(100)
acct.deposit(50)
acct.withdraw(30)
print(acct.balance)
```

A method is a plain function stored on the class. `acct.deposit(50)` calls it as `deposit(acct, 50)`, so after decoration the wrapper receives `acct` as `self`. Naming it `self` lets the wrapper ask for the class name, and passing `self` on means the original method still gets its object.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import wraps

def log_method(func):
    @wraps(func)
    def wrapper(self, *args, **kwargs):
        shown = ", ".join(map(repr, args))
        print(f"{type(self).__name__}.{func.__name__}({shown})")
        return func(self, *args, **kwargs)
    return wrapper

class Account:
    def __init__(self, balance):
        self.balance = balance

    @log_method
    def deposit(self, amount):
        self.balance += amount

    @log_method
    def withdraw(self, amount):
        self.balance -= amount

acct = Account(100)
acct.deposit(50)
acct.withdraw(30)
print(acct.balance)
```

`type(self).__name__` (not a hard-coded `"Account"`) means a subclass like `SavingsAccount` logs its own name. `func.__qualname__` is another option: it already holds `Account.deposit`, but it shows the class where the method was *defined*, not the object's class.

</details>

In [ ]:
# your code here


### NB23-Q17 · L3 · Class decorators · Write

A decorator can also take a **class**. Write `auto_repr(cls)` that adds a
`__repr__` method to the class and returns the class. The repr shows the
class name and every instance attribute as `name=repr(value)`. Use it on
`Product`, then print a product.

```py
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price
```

`print(Product("Lamp", 24.5))` should show:

```text
Product(name='Lamp', price=24.5)
```

<details><summary><b>Hint</b></summary>

Define a function `__repr__(self)` inside `auto_repr`, then attach it with `cls.__repr__ = __repr__`. `vars(self)` gives the instance attributes as a dict.

</details>

<details><summary><b>Expected output</b></summary>

```text
Product(name='Lamp', price=24.5)
```

</details>

<details><summary><b>Solution</b></summary>

```python
def auto_repr(cls):
    def __repr__(self):
        fields = ", ".join(f"{key}={value!r}" for key, value in vars(self).items())
        return f"{type(self).__name__}({fields})"
    cls.__repr__ = __repr__
    return cls

@auto_repr
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

print(Product("Lamp", 24.5))
```

`@auto_repr` above `class Product` means `Product = auto_repr(Product)`. The decorator changes the class in place by adding a method, then returns the same class. `vars(self)` is the instance's `__dict__`, which keeps attributes in the order `__init__` set them.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass

@dataclass
class Product:
    name: str
    price: float

print(Product("Lamp", 24.5))
```

`@dataclass` (NB21) is the standard library's class decorator for exactly this, plus `__init__` and `__eq__`. Writing `auto_repr` yourself shows there's no magic: it's a function that receives a class and adds methods to it.

</details>

In [ ]:
# your code here


### NB23-Q18 · L3 · Registration decorators · Write

Build a tiny command system. Write a decorator `command` that stores the
function in the dict `COMMANDS` under the function's name and returns the
function **unchanged**. Register two functions: `hello()` returning
`"Hello there!"` and `version()` returning `"v1.4.2"`. Then, for each name in `["hello", "version", "quit"]`,
print the command's result, or `unknown command: quit` if it isn't
registered.

```text
Hello there!
v1.4.2
unknown command: quit
```

**Starting code (already in the cell below):**

```python
COMMANDS = {}
```

<details><summary><b>Hint</b></summary>

The decorator does its work at definition time (Q02): one dict assignment, then `return func`. No wrapper needed.

</details>

<details><summary><b>Expected output</b></summary>

```text
Hello there!
v1.4.2
unknown command: quit
```

</details>

<details><summary><b>Solution</b></summary>

```python
def command(func):
    COMMANDS[func.__name__] = func
    return func

@command
def hello():
    return "Hello there!"

@command
def version():
    return "v1.4.2"

for name in ["hello", "version", "quit"]:
    if name in COMMANDS:
        print(COMMANDS[name]())
    else:
        print(f"unknown command: {name}")
```

A registration decorator doesn't wrap anything. It records the function in a registry and hands it straight back, so `hello()` still works as normal. Because decorators run when the `def` runs, the registry fills up as the code is loaded.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def command(func):
    COMMANDS[func.__name__] = func
    return func

@command
def hello():
    return "Hello there!"

@command
def version():
    return "v1.4.2"

for name in ["hello", "version", "quit"]:
    handler = COMMANDS.get(name)
    print(handler() if handler else f"unknown command: {name}")
```

This is how plugin systems, CLI frameworks and web routers (the mini-project) find their handlers. `dict.get` looks the name up once instead of twice. A real registry would also refuse duplicate names rather than silently overwriting.

</details>

In [ ]:
COMMANDS = {}

# your code here


### NB23-Q19 · L3 · lru_cache and cache_info · Predict

An LRU cache that holds only **two** results. The function prints when it
really computes. Predict the output.

**What does this print?**

```python
from functools import lru_cache

@lru_cache(maxsize=2)
def shipping_rate(country):
    print("computing", country)
    return len(country) * 5

shipping_rate("UK")
shipping_rate("FR")
shipping_rate("UK")
shipping_rate("DE")
shipping_rate("FR")
print(shipping_rate.cache_info())
```

<details><summary><b>Hint</b></summary>

LRU means *least recently used*. When a third result arrives, the one that hasn't been used for longest is thrown out. Reading `UK` again counts as a use.

</details>

<details><summary><b>Expected output</b></summary>

```text
computing UK
computing FR
computing DE
computing FR
CacheInfo(hits=1, misses=4, maxsize=2, currsize=2)
```

</details>

<details><summary><b>Solution</b></summary>

`UK` and `FR` are computed (2 misses). `UK` again is a hit, which also makes `FR` the least recently used. `DE` is a miss and pushes `FR` out. So `FR` must be computed again: 4 misses and 1 hit, with 2 results stored. `cache_info()` reports exactly those numbers.

</details>

<details><summary><b>Senior dev solution</b></summary>

`cache_info()` is how seniors check a cache is earning its keep: a low hit rate means the `maxsize` is too small or the inputs never repeat. `shipping_rate.cache_clear()` empties it, for example after the price table changes.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB23-Q20 · L3 · lru_cache and cache_info · Error

A cached function receives a list. Which error does this raise?

**Which error does this raise?**

```python
from functools import cache

@cache
def basket_total(prices):
    return sum(prices)

print(basket_total([4.5, 2.0, 3.5]))
```

<details><summary><b>Hint</b></summary>

A cache is a dict behind the scenes. What does a dict require of its keys (NB06)?

</details>

<details><summary><b>Expected output</b></summary>

```text
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError`: the cache stores results in a dict keyed by the arguments, and a list is unhashable because it can change. The error happens on the call, before `basket_total` even runs.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import cache

@cache
def basket_total(prices):
    return sum(prices)

print(basket_total((4.5, 2.0, 3.5)))
```

Pass a tuple instead, or convert inside a thin wrapper: `basket_total(tuple(prices))`. Only cache functions whose arguments are hashable *and* whose result depends on nothing but those arguments.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB23-Q21 · L3 · singledispatch · Write

Use `functools.singledispatch` to write `describe(value)`, which picks
an implementation by the **type** of its argument:

- default: `other: <repr>`
- `int`: `int with <n> digits` (count the digits of `abs(value)`)
- `str`: `text of length <n>`
- `list`: `list of <n> items`

Print `describe(v)` for each `v` in `[2024, "hello", [1, 2, 3], 4.5, True]`.

```text
int with 4 digits
text of length 5
list of 3 items
other: 4.5
int with 1 digits
```

<details><summary><b>Hint</b></summary>

Decorate the default with `@singledispatch`. Then add each type with `@describe.register(int)` on a new function (the name can be `_`). Why does `True` count as an `int`? (NB01)

</details>

<details><summary><b>Expected output</b></summary>

```text
int with 4 digits
text of length 5
list of 3 items
other: 4.5
int with 1 digits
```

</details>

<details><summary><b>Solution</b></summary>

```python
from functools import singledispatch

@singledispatch
def describe(value):
    return f"other: {value!r}"

@describe.register(int)
def _(value):
    return f"int with {len(str(abs(value)))} digits"

@describe.register(str)
def _(value):
    return f"text of length {len(value)}"

@describe.register(list)
def _(value):
    return f"list of {len(value)} items"

for v in [2024, "hello", [1, 2, 3], 4.5, True]:
    print(describe(v))
```

`singledispatch` turns `describe` into a dispatcher. `.register(int)` is a decorator factory that records a function for one type. On each call it looks at the type of the **first** argument and picks the closest match, falling back to the default. `bool` is a subclass of `int`, so `True` uses the `int` version: `abs(True)` is `1`, which has 1 digit.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import singledispatch

@singledispatch
def describe(value):
    return f"other: {value!r}"

@describe.register
def _(value: int):
    return f"int with {len(str(abs(value)))} digits"

@describe.register
def _(value: str):
    return f"text of length {len(value)}"

@describe.register
def _(value: list):
    return f"list of {len(value)} items"

for v in [2024, "hello", [1, 2, 3], 4.5, True]:
    print(describe(v))
```

`register` can also read the type from the annotation, which keeps the type next to the parameter. Seniors reach for `singledispatch` when they need per-type behaviour for types they don't own (`int`, `list`); for their own classes, a method is usually simpler. Registering `bool` too would stop `True` being described as an `int`.

</details>

In [ ]:
# your code here


### NB23-Q22 · L3 · Writing a decorator · Refactor

Three report functions repeat the same start and end lines. Move that
repeated code into one decorator `logged` and use it on all three.
The output must not change.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
def sales_report():
    print("start: sales_report")
    result = "sales 1200"
    print("end: sales_report")
    return result

def stock_report():
    print("start: stock_report")
    result = "stock 85"
    print("end: stock_report")
    return result

def staff_report():
    print("start: staff_report")
    result = "staff 12"
    print("end: staff_report")
    return result

print(sales_report())
print(stock_report())
print(staff_report())
```

<details><summary><b>Hint</b></summary>

The wrapper prints `start:` with `func.__name__`, calls the function, prints `end:`, and returns the result.

</details>

<details><summary><b>Expected output</b></summary>

```text
start: sales_report
end: sales_report
sales 1200
start: stock_report
end: stock_report
stock 85
start: staff_report
end: staff_report
staff 12
```

</details>

<details><summary><b>Solution</b></summary>

```python
def logged(func):
    def wrapper():
        print("start:", func.__name__)
        result = func()
        print("end:", func.__name__)
        return result
    return wrapper

@logged
def sales_report():
    return "sales 1200"

@logged
def stock_report():
    return "stock 85"

@logged
def staff_report():
    return "staff 12"

print(sales_report())
print(stock_report())
print(staff_report())
```

Code that runs *around* a function (before and after it) is what decorators are for. Each report now holds only its own logic, and changing the log format means editing one place instead of three. The name comes from `func.__name__`, so it can never get out of sync with the function.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import wraps

def logged(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"start: {func.__name__}")
        try:
            return func(*args, **kwargs)
        finally:
            print(f"end: {func.__name__}")
    return wrapper

@logged
def sales_report():
    return "sales 1200"

@logged
def stock_report():
    return "stock 85"

@logged
def staff_report():
    return "staff 12"

print(sales_report())
print(stock_report())
print(staff_report())
```

`try`/`finally` makes the `end:` line appear even when a report crashes, which is when you most want it in the log. Anything with a guaranteed "after" step is also a candidate for a context manager (NB24).

</details>

In [ ]:
def sales_report():
    print("start: sales_report")
    result = "sales 1200"
    print("end: sales_report")
    return result

def stock_report():
    print("start: stock_report")
    result = "stock 85"
    print("end: stock_report")
    return result

def staff_report():
    print("start: staff_report")
    result = "staff 12"
    print("end: staff_report")
    return result

print(sales_report())
print(stock_report())
print(staff_report())


---

## L4 · Somewhat Difficult

### NB23-Q23 · L4 · Timing decorators · Write

Write a decorator factory `timed(clock)`. Its wrapper reads `clock()`
before and after calling the function and prints
`<name> took <seconds>s` with 2 decimal places, then returns the result.
Use it as `@timed(fake_clock)` on `build_report()` (returns
`"report ready"`) and `send_email()` (returns `"sent"`), and print each
call's result.

`fake_clock` returns the next made-up time each time it's called, so the
output never changes:

```text
build_report took 2.50s
report ready
send_email took 0.25s
sent
```

**Starting code (already in the cell below):**

```python
readings = iter([100.0, 102.5, 300.0, 300.25])

def fake_clock():
    return next(readings)
```

<details><summary><b>Hint</b></summary>

Three levels again: `timed(clock)` → `decorator(func)` → `wrapper(*args, **kwargs)`. Save `clock()` in `start`, call the function, then subtract.

</details>

<details><summary><b>Expected output</b></summary>

```text
build_report took 2.50s
report ready
send_email took 0.25s
sent
```

</details>

<details><summary><b>Solution</b></summary>

```python
def timed(clock):
    def decorator(func):
        def wrapper(*args, **kwargs):
            start = clock()
            result = func(*args, **kwargs)
            print(f"{func.__name__} took {clock() - start:.2f}s")
            return result
        return wrapper
    return decorator

@timed(fake_clock)
def build_report():
    return "report ready"

@timed(fake_clock)
def send_email():
    return "sent"

print(build_report())
print(send_email())
```

The clock is a parameter, so the decorator doesn't care whether it's real or fake. Each call reads two times: `102.5 - 100.0 = 2.5` for the report and `300.25 - 300.0 = 0.25` for the email. `:.2f` formats both with two decimal places.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import time
from functools import wraps

def timed(clock=time.perf_counter):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            start = clock()
            try:
                return func(*args, **kwargs)
            finally:
                print(f"{func.__name__} took {clock() - start:.2f}s")
        return wrapper
    return decorator

@timed(clock=fake_clock)
def build_report():
    return "report ready"

@timed(clock=fake_clock)
def send_email():
    return "sent"

print(build_report())
print(send_email())
```

The real clock is the default (`time.perf_counter`, NB15, the right clock for measuring durations), and tests pass a fake one. This is *dependency injection*: hand a function the things it depends on, and it becomes easy to test. `finally` also times calls that crash.

</details>

In [ ]:
readings = iter([100.0, 102.5, 300.0, 300.25])

def fake_clock():
    return next(readings)

# your code here


### NB23-Q24 · L4 · Validating with inspect.signature · Write

Write a decorator `validate` that checks every argument against the
function's **type annotations** before calling it. If a value isn't an
instance of its annotated type, raise `TypeError` with the message
`<param> must be <type name>, got <actual type name>`.

Use it on:

```py
def add_stock(item: str, qty: int, price: float):
    return f"{qty} x {item} at {price}"
```

Then print `add_stock("pen", 10, 1.5)`, and try the two calls
`add_stock("pen", "10", 1.5)` and `add_stock(item="pen", qty=10, price=2)`,
printing `TypeError: <message>` for each.

```text
10 x pen at 1.5
TypeError: qty must be int, got str
TypeError: price must be float, got int
```

<details><summary><b>Hint</b></summary>

`inspect.signature(func)` describes the parameters. Its `.bind(*args, **kwargs)` matches the call's values to parameter names, whether they were passed by position or by keyword. `.arguments` gives you name → value, and `sig.parameters[name].annotation` gives the type.

</details>

<details><summary><b>Expected output</b></summary>

```text
10 x pen at 1.5
TypeError: qty must be int, got str
TypeError: price must be float, got int
```

</details>

<details><summary><b>Solution</b></summary>

```python
import inspect

def validate(func):
    sig = inspect.signature(func)
    def wrapper(*args, **kwargs):
        bound = sig.bind(*args, **kwargs)
        for name, value in bound.arguments.items():
            expected = sig.parameters[name].annotation
            if not isinstance(value, expected):
                raise TypeError(f"{name} must be {expected.__name__}, "
                                f"got {type(value).__name__}")
        return func(*args, **kwargs)
    return wrapper

@validate
def add_stock(item: str, qty: int, price: float):
    return f"{qty} x {item} at {price}"

print(add_stock("pen", 10, 1.5))
try:
    add_stock("pen", "10", 1.5)
except TypeError as e:
    print("TypeError:", e)
try:
    add_stock(item="pen", qty=10, price=2)
except TypeError as e:
    print("TypeError:", e)
```

`sig.bind` does the hard part: it lines the call's values up with parameter names exactly the way Python would, so positional and keyword calls are checked the same way. The signature is worked out once, at decoration time, not on every call. `2` fails the `float` check because an `int` is not an instance of `float`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import inspect
from functools import wraps

def validate(func):
    sig = inspect.signature(func)
    @wraps(func)
    def wrapper(*args, **kwargs):
        bound = sig.bind(*args, **kwargs)
        for name, value in bound.arguments.items():
            expected = sig.parameters[name].annotation
            if expected is inspect.Parameter.empty:
                continue
            if not isinstance(value, expected):
                raise TypeError(f"{name} must be {expected.__name__}, "
                                f"got {type(value).__name__}")
        return func(*args, **kwargs)
    return wrapper

@validate
def add_stock(item: str, qty: int, price: float):
    return f"{qty} x {item} at {price}"

print(add_stock("pen", 10, 1.5))
for kwargs in ({"item": "pen", "qty": "10", "price": 1.5},
               {"item": "pen", "qty": 10, "price": 2}):
    try:
        add_stock(**kwargs)
    except TypeError as e:
        print(f"TypeError: {e}")
```

The senior version skips parameters with no annotation (`inspect.Parameter.empty`) instead of crashing on them. This only works for plain classes; annotations like `list[int]` (NB26) need a real validation library such as pydantic. Many teams accept `int` where `float` is annotated, as type checkers do.

</details>

In [ ]:
# your code here


### NB23-Q25 · L4 · Decorators written as classes · Write

Write the counting decorator as a **class**. `CountCalls(func)` stores the
function and a `calls` counter starting at 0. Calling the instance adds
1 to `calls` and then calls the stored function. Use
`functools.update_wrapper(self, func)` in `__init__` so the instance
keeps the function's name.

Decorate `ping()` (returns `"pong"`), call it three times printing each
result, then print `ping.calls` and `ping.__name__`.

```text
pong
pong
pong
3 ping
```

<details><summary><b>Hint</b></summary>

`@CountCalls` means `ping = CountCalls(ping)`, which runs `__init__`. Calling `ping()` afterwards calls the instance, which runs its `__call__` method (NB21).

</details>

<details><summary><b>Expected output</b></summary>

```text
pong
pong
pong
3 ping
```

</details>

<details><summary><b>Solution</b></summary>

```python
import functools

class CountCalls:
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.calls = 0

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.func(*args, **kwargs)

@CountCalls
def ping():
    return "pong"

print(ping())
print(ping())
print(ping())
print(ping.calls, ping.__name__)
```

Anything callable can be a decorator. Here `@CountCalls` builds an instance, and the instance's `__call__` plays the role of the wrapper. The state (`calls`) is an ordinary attribute, which is easier to read and reset than a `nonlocal` counter. `update_wrapper` is what `@wraps` uses inside, and it copies `__name__` onto the instance.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import functools

class CountCalls:
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.calls = 0

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.func(*args, **kwargs)

    def reset(self):
        self.calls = 0

@CountCalls
def ping():
    return "pong"

for _ in range(3):
    print(ping())
print(ping.calls, ping.__name__)
```

A class decorator earns its place when the decorator carries state and behaviour (`reset()`, `calls`). For a plain wrapper, a function is shorter. Watch out on methods: Q26 shows the catch.

</details>

In [ ]:
# your code here


### NB23-Q26 · L4 · Decorators written as classes · Error

The class-based `CountCalls` from Q25 works on plain functions. Here it
decorates a **method**. Which error does this raise?

**Which error does this raise?**

```python
class CountCalls:
    def __init__(self, func):
        self.func = func
        self.calls = 0

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.func(*args, **kwargs)

class Account:
    def __init__(self, balance):
        self.balance = balance

    @CountCalls
    def deposit(self, amount):
        self.balance += amount

acct = Account(100)
acct.deposit(50)
print(acct.balance)
```

<details><summary><b>Hint</b></summary>

`Account.deposit` is now a `CountCalls` object, not a function. When you write `acct.deposit(50)`, does anything pass `acct` in as `self`?

</details>

<details><summary><b>Expected output</b></summary>

```text
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError`: a plain function stored on a class becomes a *bound method* when you look it up on an instance, which is how `self` gets filled in. A `CountCalls` instance doesn't get that treatment, so `__call__` receives only `(50,)` and calls `deposit(50)`, which is missing `amount`. Function decorators (Q16) don't have this problem.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import wraps

def count_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

class Account:
    def __init__(self, balance):
        self.balance = balance

    @count_calls
    def deposit(self, amount):
        self.balance += amount

acct = Account(100)
acct.deposit(50)
print(acct.balance, Account.deposit.calls)
```

The fix most seniors pick is a function decorator, as here. The other fix is to give the class a `__get__` method so it binds like a function; that is the descriptor protocol, covered in NB25.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB23-Q27 · L4 · Decorator factories · Write

Write a decorator `log` that works **both** ways: as plain `@log` and as
`@log(prefix="AUDIT")`. The wrapper prints `<prefix>: <function name>`
and then calls the function. The default prefix is `LOG`.

Decorate `save()` with `@log`, `export()` with `@log(prefix="AUDIT")`
and `purge()` with `@log()`. Each returns nothing. Call all three.

```text
LOG: save
AUDIT: export
LOG: purge
```

<details><summary><b>Hint</b></summary>

Give `log` the signature `log(func=None, *, prefix="LOG")`. With plain `@log`, `func` is the function. With brackets, `func` is `None`, so return a decorator instead.

</details>

<details><summary><b>Expected output</b></summary>

```text
LOG: save
AUDIT: export
LOG: purge
```

</details>

<details><summary><b>Solution</b></summary>

```python
def log(func=None, *, prefix="LOG"):
    def decorator(f):
        def wrapper(*args, **kwargs):
            print(f"{prefix}: {f.__name__}")
            return f(*args, **kwargs)
        return wrapper
    if func is None:
        return decorator
    return decorator(func)

@log
def save():
    pass

@log(prefix="AUDIT")
def export():
    pass

@log()
def purge():
    pass

save()
export()
purge()
```

Plain `@log` calls `log(save)`: `func` is the function, so it is decorated at once. `@log(prefix="AUDIT")` and `@log()` call `log` with no function, so `func` is `None` and `log` returns `decorator`, which Python then applies. The `*` makes `prefix` keyword-only (NB09), so `@log("AUDIT")` can't sneak a string in as `func`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import wraps

def log(func=None, *, prefix="LOG"):
    def decorator(f):
        @wraps(f)
        def wrapper(*args, **kwargs):
            print(f"{prefix}: {f.__name__}")
            return f(*args, **kwargs)
        return wrapper
    return decorator if func is None else decorator(func)

@log
def save():
    pass

@log(prefix="AUDIT")
def export():
    pass

@log()
def purge():
    pass

save()
export()
purge()
```

This `func=None, *, options` pattern is how `dataclass` and `lru_cache` accept both `@dataclass` and `@dataclass(frozen=True)`. It removes the brackets trap from Q14 and Q15 for everyone who uses your decorator.

</details>

In [ ]:
# your code here


### NB23-Q28 · L4 · Stacking order · Fix

The rule: **every** attempt to delete a user must be logged, including
the refused ones. Only `admin` may delete. Right now the refused attempt
by `sam` never reaches the log. Fix it so the output is:

```text
log: delete_user by admin
bob deleted
log: delete_user by sam
refused: sam is not an admin
```

**Buggy code (copied into the cell below):**

```python
from functools import wraps

def log_attempt(func):
    @wraps(func)
    def wrapper(user, target):
        print(f"log: {func.__name__} by {user}")
        return func(user, target)
    return wrapper

def admin_only(func):
    @wraps(func)
    def wrapper(user, target):
        if user != "admin":
            print(f"refused: {user} is not an admin")
            return None
        return func(user, target)
    return wrapper

@admin_only
@log_attempt
def delete_user(user, target):
    print(f"{target} deleted")

delete_user("admin", "bob")
delete_user("sam", "kim")
```

<details><summary><b>Hint</b></summary>

A caller meets the **top** decorator first. Which check should a refused call pass through before it gets turned away?

</details>

<details><summary><b>Expected output</b></summary>

```text
log: delete_user by admin
bob deleted
log: delete_user by sam
refused: sam is not an admin
```

</details>

<details><summary><b>Solution</b></summary>

```python
from functools import wraps

def log_attempt(func):
    @wraps(func)
    def wrapper(user, target):
        print(f"log: {func.__name__} by {user}")
        return func(user, target)
    return wrapper

def admin_only(func):
    @wraps(func)
    def wrapper(user, target):
        if user != "admin":
            print(f"refused: {user} is not an admin")
            return None
        return func(user, target)
    return wrapper

@log_attempt
@admin_only
def delete_user(user, target):
    print(f"{target} deleted")

delete_user("admin", "bob")
delete_user("sam", "kim")
```

With `admin_only` on top, a refused call stops in the outer layer and never reaches `log_attempt` underneath. Moving `log_attempt` to the top makes it the first layer every call passes through. Both decorators are correct on their own; only the order was wrong.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import wraps

def log_attempt(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        user = args[0] if args else kwargs["user"]
        print(f"log: {func.__name__} by {user}")
        return func(*args, **kwargs)
    return wrapper

def admin_only(func):
    @wraps(func)
    def wrapper(user, *args, **kwargs):
        if user != "admin":
            print(f"refused: {user} is not an admin")
            return None
        return func(user, *args, **kwargs)
    return wrapper

@log_attempt   # outermost: sees every call, even refused ones
@admin_only
def delete_user(user, target):
    print(f"{target} deleted")

delete_user("admin", "bob")
delete_user("sam", "kim")
```

When order matters, seniors leave a short comment on the stack so nobody "tidies" it later. `@wraps` matters here too: without it, `log_attempt` would log the name `wrapper` from the layer below. Real code would raise `PermissionError` rather than return `None`.

</details>

In [ ]:
from functools import wraps

def log_attempt(func):
    @wraps(func)
    def wrapper(user, target):
        print(f"log: {func.__name__} by {user}")
        return func(user, target)
    return wrapper

def admin_only(func):
    @wraps(func)
    def wrapper(user, target):
        if user != "admin":
            print(f"refused: {user} is not an admin")
            return None
        return func(user, target)
    return wrapper

@admin_only
@log_attempt
def delete_user(user, target):
    print(f"{target} deleted")

delete_user("admin", "bob")
delete_user("sam", "kim")


### NB23-Q29 · L4 · Timing decorators · Write

Write a decorator factory `rate_limit(max_calls, per, clock)`. The wrapper
allows at most `max_calls` calls in any `per` seconds. A call that would
go over the limit does **not** run the function; it returns
`"blocked"`. Use `clock()` for the time.

Decorate `send_sms(to)` (returns `f"sent to {to}"`) with
`@rate_limit(max_calls=2, per=10, clock=clock)`, and run this script:

```py
print(send_sms("ann"))
clock.advance(1)
print(send_sms("bob"))
clock.advance(1)
print(send_sms("cy"))
clock.advance(9)
print(send_sms("dee"))
```

```text
sent to ann
sent to bob
blocked
sent to dee
```

**Starting code (already in the cell below):**

```python
class FakeClock:
    def __init__(self):
        self.now = 0.0

    def __call__(self):
        return self.now

    def advance(self, seconds):
        self.now += seconds

clock = FakeClock()
```

<details><summary><b>Hint</b></summary>

Keep a list of the times of recent *successful* calls in the closure. On each call, first drop the times older than `now - per`, then check how many are left.

</details>

<details><summary><b>Expected output</b></summary>

```text
sent to ann
sent to bob
blocked
sent to dee
```

</details>

<details><summary><b>Solution</b></summary>

```python
def rate_limit(max_calls, per, clock):
    def decorator(func):
        recent = []
        def wrapper(*args, **kwargs):
            now = clock()
            recent[:] = [t for t in recent if t > now - per]
            if len(recent) >= max_calls:
                return "blocked"
            recent.append(now)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@rate_limit(max_calls=2, per=10, clock=clock)
def send_sms(to):
    return f"sent to {to}"

print(send_sms("ann"))
clock.advance(1)
print(send_sms("bob"))
clock.advance(1)
print(send_sms("cy"))
clock.advance(9)
print(send_sms("dee"))
```

Calls happen at 0, 1, 2 and 11. At time 2 the window holds 0 and 1, so `cy` is blocked (and not recorded). At time 11 the cut-off is `11 - 10 = 1`, so `0` and `1` are both dropped (only times *after* the cut-off stay) and `dee` goes through. `recent[:] = ...` replaces the list's contents in place, so no `nonlocal` is needed.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import deque
from functools import wraps

def rate_limit(max_calls, per, clock):
    def decorator(func):
        recent = deque()
        @wraps(func)
        def wrapper(*args, **kwargs):
            now = clock()
            while recent and recent[0] <= now - per:
                recent.popleft()
            if len(recent) >= max_calls:
                return "blocked"
            recent.append(now)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@rate_limit(max_calls=2, per=10, clock=clock)
def send_sms(to):
    return f"sent to {to}"

print(send_sms("ann"))
clock.advance(1)
print(send_sms("bob"))
clock.advance(1)
print(send_sms("cy"))
clock.advance(9)
print(send_sms("dee"))
```

Times arrive in order, so the oldest is always at the front. A `deque` (NB15) drops old entries from the left in O(1) instead of rebuilding the list. A `FakeClock` with `advance()` is a common test helper: it makes "wait 9 seconds" instant and exact.

</details>

In [ ]:
class FakeClock:
    def __init__(self):
        self.now = 0.0

    def __call__(self):
        return self.now

    def advance(self, seconds):
        self.now += seconds

clock = FakeClock()

# your code here


### NB23-Q30 · L4 · Writing a decorator · Fix

`memoize` caches results, but `price(100, discount=0.25)` wrongly
returns the cached answer for `price(100)`. Fix the cache key so the
output is:

```text
100.0
75.0
75.0
```

**Buggy code (copied into the cell below):**

```python
from functools import wraps

def memoize(func):
    cache = {}
    @wraps(func)
    def wrapper(*args, **kwargs):
        if args not in cache:
            cache[args] = func(*args, **kwargs)
        return cache[args]
    return wrapper

@memoize
def price(amount, discount=0.0):
    return round(amount * (1 - discount), 2)

print(price(100))
print(price(100, discount=0.25))
print(price(100, 0.25))
```

<details><summary><b>Hint</b></summary>

Which part of the call does the key `args` leave out? The key must be hashable, and a dict isn't, but a sorted tuple of its items is.

</details>

<details><summary><b>Expected output</b></summary>

```text
100.0
75.0
75.0
```

</details>

<details><summary><b>Solution</b></summary>

```python
from functools import wraps

def memoize(func):
    cache = {}
    @wraps(func)
    def wrapper(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache:
            cache[key] = func(*args, **kwargs)
        return cache[key]
    return wrapper

@memoize
def price(amount, discount=0.0):
    return round(amount * (1 - discount), 2)

print(price(100))
print(price(100, discount=0.25))
print(price(100, 0.25))
```

The key ignored keyword arguments, so `price(100)` and `price(100, discount=0.25)` both used the key `(100,)`. The second call got the first call's answer. Adding the keyword arguments as a sorted tuple makes them part of the key; sorting means `a=1, b=2` and `b=2, a=1` match.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import lru_cache

@lru_cache(maxsize=None)
def price(amount, discount=0.0):
    return round(amount * (1 - discount), 2)

print(price(100))
print(price(100, discount=0.25))
print(price(100, 0.25))
```

Seniors use `lru_cache` and don't hand-roll caches. It gets the key right, is thread-safe, and adds `cache_info()`. It also treats `price(100, 0.25)` and `price(100, discount=0.25)` as different keys, which costs an extra miss but never a wrong answer.

</details>

In [ ]:
from functools import wraps

def memoize(func):
    cache = {}
    @wraps(func)
    def wrapper(*args, **kwargs):
        if args not in cache:
            cache[args] = func(*args, **kwargs)
        return cache[args]
    return wrapper

@memoize
def price(amount, discount=0.0):
    return round(amount * (1 - discount), 2)

print(price(100))
print(price(100, discount=0.25))
print(price(100, 0.25))


### NB23-Q31 · L4 · singledispatch · Refactor

`fmt` formats values for a report with a growing `isinstance` chain.
Refactor it into a `functools.singledispatch` function with one
registered implementation per type. Keep the output identical.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
def fmt(value):
    if isinstance(value, bool):
        return "yes" if value else "no"
    elif isinstance(value, int):
        return f"{value:,}"
    elif isinstance(value, float):
        return f"{value:.2f}"
    elif isinstance(value, list):
        return " | ".join(fmt(v) for v in value)
    else:
        return str(value)

for v in [1250000, 3.14159, True, [10, 2.5, False], "n/a"]:
    print(fmt(v))
```

<details><summary><b>Hint</b></summary>

Register `bool`, `int`, `float` and `list` separately. Does the *order* you register them in still matter, the way the `elif` order did?

</details>

<details><summary><b>Expected output</b></summary>

```text
1,250,000
3.14
yes
10 | 2.50 | no
n/a
```

</details>

<details><summary><b>Solution</b></summary>

```python
from functools import singledispatch

@singledispatch
def fmt(value):
    return str(value)

@fmt.register(bool)
def _(value):
    return "yes" if value else "no"

@fmt.register(int)
def _(value):
    return f"{value:,}"

@fmt.register(float)
def _(value):
    return f"{value:.2f}"

@fmt.register(list)
def _(value):
    return " | ".join(fmt(v) for v in value)

for v in [1250000, 3.14159, True, [10, 2.5, False], "n/a"]:
    print(fmt(v))
```

In the `if` chain, `bool` had to come before `int`, because `True` is also an `int`. Put them the other way round and `True` would print `1`. `singledispatch` always picks the **most specific** registered type, so order no longer matters, and adding a type means adding a function instead of editing a long chain.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import singledispatch

@singledispatch
def fmt(value) -> str:
    return str(value)

@fmt.register
def _(value: bool) -> str:
    return "yes" if value else "no"

@fmt.register
def _(value: int) -> str:
    return f"{value:,}"

@fmt.register
def _(value: float) -> str:
    return f"{value:.2f}"

@fmt.register
def _(value: list) -> str:
    return " | ".join(map(fmt, value))

for v in [1250000, 3.14159, True, [10, 2.5, False], "n/a"]:
    print(fmt(v))
```

Annotation-based `register` keeps each type next to its code. Another module can add `@fmt.register` for its own types (say `Decimal` or `date`) without touching this file: that is the *open/closed* idea, and a long `if` chain can't offer it.

</details>

In [ ]:
def fmt(value):
    if isinstance(value, bool):
        return "yes" if value else "no"
    elif isinstance(value, int):
        return f"{value:,}"
    elif isinstance(value, float):
        return f"{value:.2f}"
    elif isinstance(value, list):
        return " | ".join(fmt(v) for v in value)
    else:
        return str(value)

for v in [1250000, 3.14159, True, [10, 2.5, False], "n/a"]:
    print(fmt(v))


---

## L5 · Hard

### NB23-Q32 · L5 · lru_cache and cache_info · Write

Write your own `memoize` decorator that copies two features of
`lru_cache`: the wrapper gets a `cache_info()` function returning the
text `hits=<h> misses=<m> size=<s>`, and a `cache_clear()` function that
empties the cache and resets both counters. Use `functools.wraps`.

Use it on a recursive `fib(n)` (`fib(0)` is 0, `fib(1)` is 1). Print
`fib(30)`, then `fib.cache_info()`, then call `fib.cache_clear()` and
print `fib.cache_info()` again.

```text
832040
hits=28 misses=31 size=31
hits=0 misses=0 size=0
```

<details><summary><b>Hint</b></summary>

Keep `cache`, `hits` and `misses` in the closure (`nonlocal` for the counters). Define `cache_info` and `cache_clear` inside the decorator too, and attach them as attributes: `wrapper.cache_info = cache_info`. Why are there so many hits when the program calls `fib` only once?

</details>

<details><summary><b>Expected output</b></summary>

```text
832040
hits=28 misses=31 size=31
hits=0 misses=0 size=0
```

</details>

<details><summary><b>Solution</b></summary>

```python
from functools import wraps

def memoize(func):
    cache = {}
    hits = 0
    misses = 0

    @wraps(func)
    def wrapper(n):
        nonlocal hits, misses
        if n in cache:
            hits += 1
        else:
            misses += 1
            cache[n] = func(n)
        return cache[n]

    def cache_info():
        return f"hits={hits} misses={misses} size={len(cache)}"

    def cache_clear():
        nonlocal hits, misses
        cache.clear()
        hits = 0
        misses = 0

    wrapper.cache_info = cache_info
    wrapper.cache_clear = cache_clear
    return wrapper

@memoize
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(30))
print(fib.cache_info())
fib.cache_clear()
print(fib.cache_info())
```

Inside `fib`, the recursive calls look up the name `fib`, which now points at the **wrapper**, so every recursive call goes through the cache too. Each of `fib(0)` to `fib(30)` is computed once (31 misses), and the second branch of each step finds its answer already stored (28 hits). `cache_clear` must use `cache.clear()` rather than `cache = {}`, or it would make a new local dict and leave the real cache full.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(30))
info = fib.cache_info()
print(f"hits={info.hits} misses={info.misses} size={info.currsize}")
fib.cache_clear()
info = fib.cache_info()
print(f"hits={info.hits} misses={info.misses} size={info.currsize}")
```

Building it once shows how `lru_cache` works: closure state, plus helper functions hung on the wrapper as attributes. In real code seniors use the built-in, which is written in C, is thread-safe and supports any hashable arguments. `cache_info()` returns a named tuple (NB15), so its fields have names.

</details>

In [ ]:
# your code here


### NB23-Q33 · L5 · functools.wraps · Predict

`count_calls` stores its counter as an attribute on the wrapper. It is
stacked under a second decorator that also uses `@wraps`. Predict the
output.

**What does this print?**

```python
from functools import wraps

def count_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

def log(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@log
@count_calls
def ping():
    return "pong"

ping()
ping()
ping()
print(ping.calls, ping.__wrapped__.calls)
```

<details><summary><b>Hint</b></summary>

`@wraps` copies more than the name: it also copies the wrapped function's attribute dict (`__dict__`), **once**, at decoration time. Which object do the calls actually increase?

</details>

<details><summary><b>Expected output</b></summary>

```text
0 3
```

</details>

<details><summary><b>Solution</b></summary>

`count_calls` runs first and makes a wrapper with `calls = 0`. Then `log` runs, and its `@wraps` copies that `__dict__` into the outer wrapper, so the outer one gets its own `calls = 0`, a **snapshot**. Every call increases the inner wrapper's `calls`, never the copy. `ping.calls` stays `0`, while `ping.__wrapped__` (the inner wrapper, which `wraps` also records) shows `3`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import wraps

def count_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

def log(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@count_calls
@log
def ping():
    return "pong"

ping()
ping()
ping()
print(ping.calls)
```

State hung on a wrapper is only reliable on the **outermost** layer. Seniors either put the stateful decorator on top, as here (prints `3`), or keep state in an object the attribute *points to* (a dict or a counter class), so every copy shares the same thing.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB23-Q34 · L5 · Decorator factories · Write

Write `retry(times, delay, exceptions, sleep)`:

- Call the function up to `times` times.
- Only errors whose type is in the tuple `exceptions` are retried. Any
  other error goes straight to the caller.
- After a failed attempt, print the line shown below and call
  `sleep(seconds)`. The wait starts at `delay` and doubles each time.
  Don't sleep after the **last** attempt.
- If every attempt fails, raise
  `RuntimeError(f"{name} failed after {times} attempts")` **from** the
  last error.

`fake_sleep` (given) prints instead of waiting. Decorate the three given
functions with `@retry(times=3, delay=1, exceptions=(ConnectionError,), sleep=fake_sleep)`,
then run the script below the expected output.

```text
fetch: attempt 1 failed (ConnectionError)
  (sleeping 1s)
fetch: attempt 2 failed (ConnectionError)
  (sleeping 2s)
fetch -> data
parse -> ValueError, not retried
ping: attempt 1 failed (ConnectionError)
  (sleeping 1s)
ping: attempt 2 failed (ConnectionError)
  (sleeping 2s)
ping: attempt 3 failed (ConnectionError)
ping -> RuntimeError: ping failed after 3 attempts (cause: ConnectionError)
```

Script:

```py
print("fetch ->", fetch())
try:
    parse()
except ValueError:
    print("parse -> ValueError, not retried")
try:
    ping()
except RuntimeError as e:
    print(f"ping -> RuntimeError: {e} (cause: {type(e.__cause__).__name__})")
```

**Starting code (already in the cell below):**

```python
def fake_sleep(seconds):
    print(f"  (sleeping {seconds}s)")

fetch_calls = [0]

def fetch():
    fetch_calls[0] += 1
    if fetch_calls[0] < 3:
        raise ConnectionError("timeout")
    return "data"

def parse():
    raise ValueError("bad json")

def ping():
    raise ConnectionError("host down")
```

<details><summary><b>Hint</b></summary>

The given functions aren't decorated yet. Apply your decorator by reassigning each name (Q03). Inside the loop, `except exceptions as e:` catches only the listed types. Keep the last error in a variable so you can `raise ... from` it after the loop.

</details>

<details><summary><b>Expected output</b></summary>

```text
fetch: attempt 1 failed (ConnectionError)
  (sleeping 1s)
fetch: attempt 2 failed (ConnectionError)
  (sleeping 2s)
fetch -> data
parse -> ValueError, not retried
ping: attempt 1 failed (ConnectionError)
  (sleeping 1s)
ping: attempt 2 failed (ConnectionError)
  (sleeping 2s)
ping: attempt 3 failed (ConnectionError)
ping -> RuntimeError: ping failed after 3 attempts (cause: ConnectionError)
```

</details>

<details><summary><b>Solution</b></summary>

```python
from functools import wraps

def retry(times, delay, exceptions, sleep):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            wait = delay
            last_error = None
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_error = e
                    print(f"{func.__name__}: attempt {attempt} failed ({type(e).__name__})")
                    if attempt < times:
                        sleep(wait)
                        wait *= 2
            raise RuntimeError(f"{func.__name__} failed after {times} attempts") from last_error
        return wrapper
    return decorator

policy = retry(times=3, delay=1, exceptions=(ConnectionError,), sleep=fake_sleep)
fetch = policy(fetch)
parse = policy(parse)
ping = policy(ping)

print("fetch ->", fetch())
try:
    parse()
except ValueError:
    print("parse -> ValueError, not retried")
try:
    ping()
except RuntimeError as e:
    print(f"ping -> RuntimeError: {e} (cause: {type(e.__cause__).__name__})")
```

`except exceptions as e` accepts a tuple of types, so the `ValueError` from `parse` isn't caught and leaves the wrapper at once. Sleeping only when `attempt < times` avoids a pointless wait before giving up. `raise ... from last_error` (NB12) sets `__cause__`, so the caller gets a clear summary error *and* the real reason. One `policy` object can decorate many functions, because each call to `policy(...)` builds a fresh wrapper.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import wraps
import time

def retry(times=3, delay=1, exceptions=(ConnectionError,), sleep=time.sleep):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"{func.__name__}: attempt {attempt} failed ({type(e).__name__})")
                    if attempt == times:
                        raise RuntimeError(
                            f"{func.__name__} failed after {times} attempts") from e
                    sleep(delay * 2 ** (attempt - 1))
        return wrapper
    return decorator

policy = retry(times=3, delay=1, exceptions=(ConnectionError,), sleep=fake_sleep)
fetch, parse, ping = policy(fetch), policy(parse), policy(ping)

print("fetch ->", fetch())
try:
    parse()
except ValueError:
    print("parse -> ValueError, not retried")
try:
    ping()
except RuntimeError as e:
    print(f"ping -> RuntimeError: {e} (cause: {type(e.__cause__).__name__})")
```

Raising from inside the `except` on the last attempt removes the `last_error` variable, and `delay * 2 ** (attempt - 1)` computes the backoff without extra state. Real `time.sleep` is the default and tests inject a fake. Production retry code also adds random *jitter* so many clients don't all retry at the same instant.

</details>

In [ ]:
def fake_sleep(seconds):
    print(f"  (sleeping {seconds}s)")

fetch_calls = [0]

def fetch():
    fetch_calls[0] += 1
    if fetch_calls[0] < 3:
        raise ConnectionError("timeout")
    return "data"

def parse():
    raise ValueError("bad json")

def ping():
    raise ConnectionError("host down")

# your code here


### NB23-Q35 · L5 · Stacking order · Write

Write `apply_all(func, decorators)` that applies a **list** of decorators
to `func` so the result behaves exactly like stacking them with `@` in
the same order (the first item in the list is the top line). Use the
given `tag` factory.

Build `menu` with `@` stacking and `menu_built` with `apply_all`, then
print both results and whether they are equal.

```text
<nav><ul><li>Home</li></ul></nav>
<nav><ul><li>Home</li></ul></nav>
True
```

**Starting code (already in the cell below):**

```python
def tag(name):
    def decorator(func):
        def wrapper():
            return f"<{name}>{func()}</{name}>"
        return wrapper
    return decorator

def item():
    return "<li>Home</li>"
```

<details><summary><b>Hint</b></summary>

Stacked decorators are applied from the **bottom** up. A plain loop over the list applies them top-down, which nests them inside out. `reversed()` (NB04) fixes that.

</details>

<details><summary><b>Expected output</b></summary>

```text
<nav><ul><li>Home</li></ul></nav>
<nav><ul><li>Home</li></ul></nav>
True
```

</details>

<details><summary><b>Solution</b></summary>

```python
def apply_all(func, decorators):
    for deco in reversed(decorators):
        func = deco(func)
    return func

@tag("nav")
@tag("ul")
def menu():
    return "<li>Home</li>"

menu_built = apply_all(item, [tag("nav"), tag("ul")])

print(menu())
print(menu_built())
print(menu() == menu_built())
```

`@tag("nav") @tag("ul") def menu` means `menu = tag("nav")(tag("ul")(menu))`: the bottom decorator touches the function first. Looping in list order would wrap `nav` first and `ul` outside it, giving `<ul><nav>...`. Looping over `reversed(decorators)` applies the last one first, matching `@`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import reduce

def apply_all(func, decorators):
    return reduce(lambda f, deco: deco(f), reversed(decorators), func)

@tag("nav")
@tag("ul")
def menu():
    return "<li>Home</li>"

menu_built = apply_all(item, [tag("nav"), tag("ul")])

print(menu())
print(menu_built())
print(menu() == menu_built())
```

`reduce` (NB10) folds the list into one function, the classic way to *compose* functions. The loop is just as good and easier to read. Frameworks use exactly this to apply a configurable list of middleware, and getting the order backwards is a well-known bug.

</details>

In [ ]:
def tag(name):
    def decorator(func):
        def wrapper():
            return f"<{name}>{func()}</{name}>"
        return wrapper
    return decorator

def item():
    return "<li>Home</li>"

# your code here


### NB23-Q36 · L5 · Validating with inspect.signature · Write

Write a decorator factory `requires_role(role)`. Each decorated function
has a parameter called `user` (a dict with `"name"` and `"role"`), which
callers may pass **by position or by keyword**. Before the call, find
`user` and raise `PermissionError(f"{name} needs role {role!r}")` if
their role doesn't match.

It must also check **at decoration time**: if the function has no `user`
parameter, raise `TypeError(f"{func.__name__} has no 'user' parameter")`.

Decorate `delete_post(post_id, user)` (returns
`f"post {post_id} deleted by {user['name']}"`) with
`@requires_role("admin")`, then run the script.

```py
print(delete_post(7, ann))
print(delete_post(post_id=8, user=ann))
try:
    delete_post(9, user=bob)
except PermissionError as e:
    print("PermissionError:", e)
try:
    @requires_role("admin")
    def ping():
        return "pong"
except TypeError as e:
    print("TypeError:", e)
```

```text
post 7 deleted by ann
post 8 deleted by ann
PermissionError: bob needs role 'admin'
TypeError: ping has no 'user' parameter
```

**Starting code (already in the cell below):**

```python
ann = {"name": "ann", "role": "admin"}
bob = {"name": "bob", "role": "editor"}
```

<details><summary><b>Hint</b></summary>

Work out `inspect.signature(func)` inside `decorator(func)`, before defining the wrapper; that's decoration time. `sig.parameters` behaves like a dict of names. In the wrapper, `sig.bind(*args, **kwargs).arguments["user"]` finds the user however it was passed.

</details>

<details><summary><b>Expected output</b></summary>

```text
post 7 deleted by ann
post 8 deleted by ann
PermissionError: bob needs role 'admin'
TypeError: ping has no 'user' parameter
```

</details>

<details><summary><b>Solution</b></summary>

```python
import inspect

def requires_role(role):
    def decorator(func):
        sig = inspect.signature(func)
        if "user" not in sig.parameters:
            raise TypeError(f"{func.__name__} has no 'user' parameter")
        def wrapper(*args, **kwargs):
            user = sig.bind(*args, **kwargs).arguments["user"]
            if user["role"] != role:
                raise PermissionError(f"{user['name']} needs role {role!r}")
            return func(*args, **kwargs)
        return wrapper
    return decorator

@requires_role("admin")
def delete_post(post_id, user):
    return f"post {post_id} deleted by {user['name']}"

print(delete_post(7, ann))
print(delete_post(post_id=8, user=ann))
try:
    delete_post(9, user=bob)
except PermissionError as e:
    print("PermissionError:", e)
try:
    @requires_role("admin")
    def ping():
        return "pong"
except TypeError as e:
    print("TypeError:", e)
```

The trap is `args[0]`: `user` might be the second positional argument, or a keyword, or both styles in one program. `sig.bind` maps the call onto the parameter names exactly as Python would, so `arguments["user"]` always finds it. Checking for the parameter inside `decorator` means a mistake shows up when the module loads, not weeks later on the first real call. A decorator can raise just like any other function.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import inspect
from functools import wraps

def requires_role(role):
    def decorator(func):
        sig = inspect.signature(func)
        if "user" not in sig.parameters:
            raise TypeError(f"{func.__name__} has no 'user' parameter")

        @wraps(func)
        def wrapper(*args, **kwargs):
            bound = sig.bind(*args, **kwargs)
            user = bound.arguments["user"]
            if user["role"] != role:
                raise PermissionError(f"{user['name']} needs role {role!r}")
            return func(*bound.args, **bound.kwargs)
        return wrapper
    return decorator

@requires_role("admin")
def delete_post(post_id, user):
    return f"post {post_id} deleted by {user['name']}"

print(delete_post(7, ann))
print(delete_post(post_id=8, user=ann))
try:
    delete_post(9, user=bob)
except PermissionError as e:
    print("PermissionError:", e)
try:
    @requires_role("admin")
    def ping():
        return "pong"
except TypeError as e:
    print("TypeError:", e)
```

"Fail at import, not at runtime" is a core senior habit: the decoration-time check turns a hidden bug into an immediate error. `bound.args` and `bound.kwargs` pass on exactly what was bound. Web frameworks implement `@login_required` and permission checks this way.

</details>

In [ ]:
ann = {"name": "ann", "role": "admin"}
bob = {"name": "bob", "role": "editor"}

# your code here


---

## Mini-project

Pull together everything from this notebook.

## NB23-P · L5 · Mini-project: Mini Web Router · Write

Build a tiny web router from decorators.

1. A `Router` class with a `routes` list. Its method `route(pattern)` is a
   decorator factory: it records `(pattern, handler)` in `routes` and
   returns the handler **unchanged**.
2. Patterns can hold path parameters in angle brackets, like
   `/users/<user_id>`. `dispatch(request)` splits the request's `"path"`
   and each pattern on `/`, finds the first pattern with the same number
   of parts where every fixed part matches, and calls
   `handler(request, **params)` (for example `user_id="42"`). If nothing
   matches it returns `"404 Not Found"`.
3. Two middleware decorators:
   - `log_request` prints `[log] <path>` before calling the handler.
   - `require_login` returns `"401 Unauthorized"` without calling the
     handler when `request["user"]` is `None`.

Register the handlers below. `@app.route(...)` must be the **top**
decorator, so the router stores the fully wrapped handler.

- `/` → `home`, logged, returns `"200 Home"`
- `/users/<user_id>` → `show_user`, logged, returns `f"200 User {user_id}"`
- `/admin/stats` → `stats`, logged **and** login required (logging
  outside), returns `f"200 Stats for {request['user']}"`

Then print `app.dispatch(r)` for each request in `requests` (given).

```text
[log] /
200 Home
[log] /users/42
200 User 42
[log] /admin/stats
401 Unauthorized
[log] /admin/stats
200 Stats for ann
404 Not Found
```

**Starting code (already in the cell below):**

```python
requests = [
    {"path": "/", "user": None},
    {"path": "/users/42", "user": None},
    {"path": "/admin/stats", "user": None},
    {"path": "/admin/stats", "user": "ann"},
    {"path": "/orders/9/items", "user": "ann"},
]
```

<details><summary><b>Hint</b></summary>

Split with `path.strip("/").split("/")`, so `/` becomes `[""]` and still matches itself. A `for`/`else` (NB04) or a flag tells you whether every part matched. Write the middleware as ordinary decorators whose wrapper takes `(request, **params)`.

</details>

<details><summary><b>Expected output</b></summary>

```text
[log] /
200 Home
[log] /users/42
200 User 42
[log] /admin/stats
401 Unauthorized
[log] /admin/stats
200 Stats for ann
404 Not Found
```

</details>

<details><summary><b>Solution</b></summary>

```python
from functools import wraps

class Router:
    def __init__(self):
        self.routes = []

    def route(self, pattern):
        def decorator(handler):
            self.routes.append((pattern, handler))
            return handler
        return decorator

    def dispatch(self, request):
        parts = request["path"].strip("/").split("/")
        for pattern, handler in self.routes:
            pattern_parts = pattern.strip("/").split("/")
            if len(pattern_parts) != len(parts):
                continue
            params = {}
            matched = True
            for expected, actual in zip(pattern_parts, parts):
                if expected.startswith("<") and expected.endswith(">"):
                    params[expected[1:-1]] = actual
                elif expected != actual:
                    matched = False
                    break
            if matched:
                return handler(request, **params)
        return "404 Not Found"

def log_request(handler):
    @wraps(handler)
    def wrapper(request, **params):
        print(f"[log] {request['path']}")
        return handler(request, **params)
    return wrapper

def require_login(handler):
    @wraps(handler)
    def wrapper(request, **params):
        if request["user"] is None:
            return "401 Unauthorized"
        return handler(request, **params)
    return wrapper

app = Router()

@app.route("/")
@log_request
def home(request):
    return "200 Home"

@app.route("/users/<user_id>")
@log_request
def show_user(request, user_id):
    return f"200 User {user_id}"

@app.route("/admin/stats")
@log_request
@require_login
def stats(request):
    return f"200 Stats for {request['user']}"

for r in requests:
    print(app.dispatch(r))
```

`@app.route("/")` calls a **method** that returns a decorator; the closure remembers both `self` and `pattern`. Because decorators apply bottom-up, the middleware wraps the handler first and `route` records the finished result. With `route` lower down, the router would store the bare handler and the middleware would never run. `log_request` sits above `require_login`, so refused requests are still logged (Q28). The last request matches nothing and never reaches a handler, so it isn't logged.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import re
from functools import wraps

class Router:
    def __init__(self):
        self.routes = []

    def route(self, pattern):
        regex = re.compile("^" + re.sub(r"<(\w+)>", r"(?P<\1>[^/]+)", pattern) + "$")
        def decorator(handler):
            self.routes.append((regex, handler))
            return handler
        return decorator

    def dispatch(self, request):
        for regex, handler in self.routes:
            if match := regex.match(request["path"]):
                return handler(request, **match.groupdict())
        return "404 Not Found"

def log_request(handler):
    @wraps(handler)
    def wrapper(request, **params):
        print(f"[log] {request['path']}")
        return handler(request, **params)
    return wrapper

def require_login(handler):
    @wraps(handler)
    def wrapper(request, **params):
        if request["user"] is None:
            return "401 Unauthorized"
        return handler(request, **params)
    return wrapper

app = Router()

@app.route("/")
@log_request
def home(request):
    return "200 Home"

@app.route("/users/<user_id>")
@log_request
def show_user(request, user_id):
    return f"200 User {user_id}"

@app.route("/admin/stats")
@log_request
@require_login
def stats(request):
    return f"200 Stats for {request['user']}"

for r in requests:
    print(app.dispatch(r))
```

The senior compiles each pattern once into a regex (NB17) with named groups, so `<user_id>` becomes `(?P<user_id>[^/]+)` and `groupdict()` hands back the parameters. Compiling at registration time (decoration time) keeps every request fast. Flask and FastAPI routes work on this same idea.

</details>

In [ ]:
requests = [
    {"path": "/", "user": None},
    {"path": "/users/42", "user": None},
    {"path": "/admin/stats", "user": None},
    {"path": "/admin/stats", "user": "ann"},
    {"path": "/orders/9/items", "user": "ann"},
]

# your code here


---

## Done

Next up: **NB24 · Context Managers** in `advanced/`.